# Pedigree System in PyPath

## Overview
The pedigree system tracks data quality and uncertainty across Ecopath parameters. This notebook demonstrates the complete pedigree workflow: creating quality levels, building containers, validating data, and integrating with the sampling framework.

In [ ]:
# Import required modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pypath.core.params import PedigreeQuality, EwePedigreeConfig, PedigreeContainer
from pypath.core.stanzas import PedigreeEntry

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Part 1: Understanding PedigreeQuality Enum

The PedigreeQuality enum defines five levels representing data confidence:

In [ ]:
# Display available pedigree quality levels
for quality in PedigreeQuality:
    print(f'{quality.name}: {quality.value}')

**Quality Levels:**
- **EXPERT** (5): Peer-reviewed research or direct measurements
- **SCIENTIFIC_PAPER** (4): Published but not peer-reviewed
- **FIELD_MEASUREMENT** (3): Direct field data collection
- **ASSUMED** (2): Reasonable biological assumptions
- **GUESSED** (1): Rough estimates or expert guesses

Lower numerical values indicate lower confidence. The mapping to CoV (Coefficient of Variation) is: `CoV = 0.1 * (6 - quality_value)`

## Part 2: Creating EwePedigreeConfig

Configure quality ratings across different parameter types:

In [ ]:
# Create a pedigree configuration
pedigree_config = EwePedigreeConfig(
    biomass_quality=PedigreeQuality.FIELD_MEASUREMENT,
    pb_quality=PedigreeQuality.SCIENTIFIC_PAPER,
    qb_quality=PedigreeQuality.ASSUMED,
    diet_quality=PedigreeQuality.EXPERT,
    dc_quality=PedigreeQuality.GUESSED
)

print('Pedigree Configuration:')
print(f'  Biomass quality: {pedigree_config.biomass_quality.name} ({pedigree_config.biomass_quality.value})')
print(f'  P/B quality: {pedigree_config.pb_quality.name}')
print(f'  Q/B quality: {pedigree_config.qb_quality.name}')
print(f'  Diet quality: {pedigree_config.diet_quality.name}')
print(f'  DC quality: {pedigree_config.dc_quality.name}')

## Part 3: Building a PedigreeContainer

The PedigreeContainer manages pedigree data for all groups in a model:

In [ ]:
# Create a pedigree container
pedigree = PedigreeContainer()

# Add groups with their pedigree configurations
groups = ['Cod', 'Herring', 'Zooplankton', 'Phytoplankton', 'Detritus']
for group in groups:
    pedigree.add_group(group, pedigree_config)

print(f'Added {len(pedigree)} groups to pedigree container')
print(f'Groups: {list(pedigree.groups())}')

## Part 4: Accessing and Modifying Pedigree Data

In [ ]:
# Get pedigree for specific group
cod_pedigree = pedigree.get_group('Cod')
print(f'Cod pedigree:')
print(f'  Biomass quality: {cod_pedigree.biomass_quality.name}')
print(f'  P/B quality: {cod_pedigree.pb_quality.name}')

# Update specific parameter quality
pedigree.set_biomass_quality('Herring', PedigreeQuality.EXPERT)
herring_pedigree = pedigree.get_group('Herring')
print(f'\nUpdated Herring biomass quality: {herring_pedigree.biomass_quality.name}')

## Part 5: DataFrame Serialization

In [ ]:
# Convert to DataFrame
pedigree_df = pedigree.to_dataframe()
print('Pedigree DataFrame:')
print(pedigree_df)

# Save to CSV
pedigree_df.to_csv('pedigree_backup.csv', index=True)
print('\nSaved to pedigree_backup.csv')

## Part 6: Integration with Sampling Framework

Convert pedigree quality to Coefficient of Variation (CoV):

In [ ]:
# Get CoV mapping
cov_mapping = pedigree.to_cov_mapping()
print('Quality to CoV Mapping:')
for quality in PedigreeQuality:
    cov = cov_mapping.get(quality)
    print(f'  {quality.name}: CoV = {cov:.3f}')

## Performance Tips

- PedigreeContainer operations are O(1) for group access
- DataFrame conversion is O(n) where n = number of groups
- Use `.to_cov_mapping()` once and reuse for sampling
- Validate pedigree once after initialization